In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from shadow_peft import ShadowForCausalLM

shadow_peft_id = 'erin99/Qwen3-4B-GSM8k-Shadow'  # '/home/lxm/workspace/Shadow/outputs/gsm8k_suite_shadow_4B/gsm8k_gsm8k_main/shadow_peft'
tokenizer = AutoTokenizer.from_pretrained(shadow_peft_id)
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B").to('cuda:7')

model = ShadowForCausalLM.from_pretrained(
    base_model, shadow_peft_id, is_trainable=False).to(base_model.device)

model = model.eval()



/home/lxm/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  3.20it/s]


In [55]:
import torch

# Example GSM8K problem
# question = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
question = "1+1=?"
question = "Ariel was playing basketball. 1 of her shots went in the hoop. 2 of her shots did not go in the hoop. How many shots were there in total?"
question = 'our team scored a total of 123 points. 67 points were scored in the first half. How many were scored in the second half?'
question = "小王的妈妈有三个儿子，大儿子叫大明，二儿子叫二明，三儿子叫什么？"
# question = "9.3 and 9.200 which is bigger?"
question = "blood relationship identify: Grand Mother's Brother"
question = '有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？'

# Format the prompt according to GSM8K format
user_content = f"Question: {question}\nAnswer:"
user_message = {"role": "user", "content": user_content}
prompt = tokenizer.apply_chat_template(
    [user_message],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

# Tokenize and generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
    )

# Decode and print the result
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


user
Question: 有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？
Answer:
assistant
<think>

</think>

彩珠的排列周期是2+3+4=<<2+3+4=9>>9颗珠子。
509÷9=56余<<509%9=5>>5颗珠子。
余数是5，说明第509颗珠子是第5个珠子。
第5个珠子是绿色。
#### 绿
